# Plant Disease Detection Using Deep Learning

This notebook contains the main training and evaluation work for the plant disease detection project.

The dataset contains 38 plant disease and healthy-leaf classes. We will use two pretrained models:

- ResNet-50
- EfficientNet-B0

The goal is to train both models, compare them fairly, keep a clean hold-out test set, and prepare the final model for Grad-CAM based explainability.

A small set of 76 images (2 images from each class) is also reserved separately for the live demonstration. Those images will not be used for training, validation, or final testing.

## 1. Basic setup

This first cell imports the libraries used throughout the notebook and checks whether Kaggle has given us a GPU.

The notebook is designed to run in Kaggle. For training the pretrained models, make sure the notebook accelerator is set to **GPU**.

In [ ]:
from pathlib import Path
import json
import random
import shutil
import hashlib
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from torchvision.models import (
    ResNet50_Weights,
    EfficientNet_B0_Weights,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

SEED = 67

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Dataset paths and project settings

The Kaggle dataset has a nested folder structure. We already checked the structure before starting the main notebook, so the path below points directly to the folder containing `train` and `valid`.

The training settings are kept together here so they are easy to change later without searching through the notebook.

In [ ]:
DATASET_ROOT = Path(
    "/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/"
    "New Plant Diseases Dataset(Augmented)/"
    "New Plant Diseases Dataset(Augmented)"
)

TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"

WORK_DIR = Path("/kaggle/working")
DEMO_DIR = WORK_DIR / "demo_images"
ARTIFACT_DIR = WORK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3

TEST_FRACTION = 0.20

print("Train directory:", TRAIN_DIR)
print("Validation directory:", VALID_DIR)
print("Working directory:", WORK_DIR)

## 3. Check the dataset again

Before training, I want the notebook itself to confirm the number of classes and the total number of images.

The counts should match the audit we already performed: 38 classes, about 70K training images and about 17.5K images in the provided validation folder.

In [ ]:
def image_files(folder):
    return sorted(
        file for file in folder.iterdir()
        if file.is_file() and file.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    )

class_names = sorted(
    folder.name for folder in TRAIN_DIR.iterdir()
    if folder.is_dir()
)

print("Number of classes:", len(class_names))
print()

train_counts = {}
valid_counts = {}

for class_name in class_names:
    train_counts[class_name] = len(image_files(TRAIN_DIR / class_name))
    valid_counts[class_name] = len(image_files(VALID_DIR / class_name))

count_table = pd.DataFrame({
    "class": class_names,
    "train": [train_counts[c] for c in class_names],
    "valid": [valid_counts[c] for c in class_names],
})

print(count_table.to_string(index=False))

print()
print("Total training images:", count_table["train"].sum())
print("Total provided validation images:", count_table["valid"].sum())

## 4. Check image readability and basic properties

We already checked a sample of images before starting the main notebook. This time the notebook checks every image in the training and validation folders.

A failed read would be reported instead of silently being ignored.

In [ ]:
def check_images(folder):
    bad_files = []
    modes = set()
    sizes = set()

    for class_dir in sorted(folder.iterdir()):
        if not class_dir.is_dir():
            continue

        for file in image_files(class_dir):
            try:
                with Image.open(file) as img:
                    img.verify()

                with Image.open(file) as img:
                    modes.add(img.mode)
                    sizes.add(img.size)

            except Exception as exc:
                bad_files.append((str(file), str(exc)))

    return bad_files, modes, sizes

start = time.time()

train_bad, train_modes, train_sizes = check_images(TRAIN_DIR)
valid_bad, valid_modes, valid_sizes = check_images(VALID_DIR)

print("Training images with read errors:", len(train_bad))
print("Validation images with read errors:", len(valid_bad))
print("Image modes found:", sorted(train_modes | valid_modes))
print("Number of different image sizes:", len(train_sizes | valid_sizes))
print("Sizes found:", sorted(train_sizes | valid_sizes)[:10])

print(f"Check completed in {time.time() - start:.1f} seconds")

if train_bad:
    print("\nFirst training error:")
    print(train_bad[0])

if valid_bad:
    print("\nFirst validation error:")
    print(valid_bad[0])

## 5. Look at a few real images

Numbers are useful, but I also want to see what the images look like.

This is only a visual sanity check. It helps us confirm that the folders contain sensible leaf images and that the disease/healthy labels look reasonable at a glance.

In [ ]:
rng = random.Random(SEED)

samples = []
for class_name in class_names:
    files = image_files(VALID_DIR / class_name)
    if files:
        samples.append((class_name, rng.choice(files)))

samples = samples[:12]

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, (class_name, file) in zip(axes.ravel(), samples):
    with Image.open(file) as img:
        ax.imshow(img.convert("RGB"))
    ax.set_title(class_name.replace("___", " - "))
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Reserve the 76 live-demo images

Two images from every class will be reserved before training.

The seed is fixed at **67**, and the files are sorted before sampling. This makes the selection repeatable. If the notebook is started again with the same dataset, it will select the same two files from each class.

These files will be copied into `/kaggle/working/demo_images/`. The original Kaggle dataset is left unchanged.

In [ ]:
if DEMO_DIR.exists():
    shutil.rmtree(DEMO_DIR)

DEMO_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

demo_records = []

for class_name in class_names:
    source_files = image_files(VALID_DIR / class_name)

    if len(source_files) < 2:
        raise RuntimeError(f"Not enough images in class: {class_name}")

    selected = random.sample(source_files, 2)

    class_demo_dir = DEMO_DIR / class_name
    class_demo_dir.mkdir(parents=True, exist_ok=True)

    for source_file in selected:
        destination = class_demo_dir / source_file.name
        shutil.copy2(source_file, destination)

        demo_records.append({
            "class_name": class_name,
            "file_name": source_file.name,
            "original_path": str(source_file),
            "demo_path": str(destination),
        })

demo_manifest = pd.DataFrame(demo_records)
demo_manifest_path = DEMO_DIR / "demo_manifest.csv"
demo_manifest.to_csv(demo_manifest_path, index=False)

print("Classes covered:", demo_manifest["class_name"].nunique())
print("Images reserved:", len(demo_manifest))
print("Manifest:", demo_manifest_path)

## 7. Create a copy of the demo set as a ZIP

This archive is only for convenience. It gives us a portable copy of the 76 images that can be downloaded from Kaggle and kept on the local PC for the final demonstration.

The model training code does not use this ZIP.

In [ ]:
demo_zip = shutil.make_archive(
    str(WORK_DIR / "demo_images"),
    "zip",
    root_dir=WORK_DIR,
    base_dir="demo_images",
)

print("Demo ZIP:", demo_zip)
print(f"ZIP size: {Path(demo_zip).stat().st_size / (1024 * 1024):.2f} MB")

## 8. Check exact duplicates across the original train/valid folders

The dataset contains many augmented images, so visually similar images are expected. Here I am checking a narrower question: are any files byte-for-byte identical across the training and validation folders?

Exact duplicates would be worth knowing before we interpret evaluation results.

In [ ]:
def file_hash(file, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

start = time.time()

train_hashes = {}

for class_name in class_names:
    for file in image_files(TRAIN_DIR / class_name):
        h = file_hash(file)
        train_hashes.setdefault(h, str(file))

duplicate_pairs = []

for class_name in class_names:
    for file in image_files(VALID_DIR / class_name):
        h = file_hash(file)
        if h in train_hashes:
            duplicate_pairs.append((train_hashes[h], str(file)))

print("Exact train/valid duplicate pairs:", len(duplicate_pairs))
print(f"Hash check completed in {time.time() - start:.1f} seconds")

if duplicate_pairs:
    print("\nFirst few exact duplicate pairs:")
    for pair in duplicate_pairs[:10]:
        print(pair)

## 9. Build the final train, validation and test lists

The original dataset gives us `train` and `valid`. We will keep the training folder as training data.

From the provided `valid` folder, we first remove the 76 demo images. The remaining images are then split into:

- validation: used while developing and selecting the model
- test: held out until the model choices are finished

This gives us a clean test subset without needing another external dataset.

In [ ]:
demo_keys = set(
    zip(demo_manifest["class_name"], demo_manifest["file_name"])
)

remaining_valid = []

for class_name in class_names:
    for file in image_files(VALID_DIR / class_name):
        if (class_name, file.name) not in demo_keys:
            remaining_valid.append((str(file), class_name))

validation_items = []
test_items = []

for class_name in class_names:
    class_items = [
        item for item in remaining_valid
        if item[1] == class_name
    ]

    class_items = sorted(class_items, key=lambda x: x[0])

    rng = random.Random(SEED)
    rng.shuffle(class_items)

    test_count = max(1, int(len(class_items) * TEST_FRACTION))

    test_items.extend(class_items[:test_count])
    validation_items.extend(class_items[test_count:])

print("Training images:", count_table["train"].sum())
print("Validation images:", len(validation_items))
print("Test images:", len(test_items))
print("Demo images:", len(demo_manifest))

print("\nTotal images used in model development/evaluation:",
      count_table["train"].sum() + len(validation_items) + len(test_items))

print("Total images including demo set:",
      count_table["train"].sum() + len(validation_items) + len(test_items) + len(demo_manifest))

## 10. Save the split information

The split lists are saved as CSV files so the experiment can be reproduced later.

This is also useful for documentation because we can always show exactly which images were assigned to training, validation and test.

In [ ]:
train_items = []

for class_name in class_names:
    for file in image_files(TRAIN_DIR / class_name):
        train_items.append((str(file), class_name))

train_manifest = pd.DataFrame(train_items, columns=["path", "class_name"])
validation_manifest = pd.DataFrame(validation_items, columns=["path", "class_name"])
test_manifest = pd.DataFrame(test_items, columns=["path", "class_name"])

train_manifest.to_csv(ARTIFACT_DIR / "train_manifest.csv", index=False)
validation_manifest.to_csv(ARTIFACT_DIR / "validation_manifest.csv", index=False)
test_manifest.to_csv(ARTIFACT_DIR / "test_manifest.csv", index=False)

print("Saved split manifests to:", ARTIFACT_DIR)

## 11. Define image preprocessing

The pretrained models were originally trained on ImageNet, so we will use ImageNet's standard normalization values.

For training, a few simple augmentations are added. They create small variations in the training images and help the model avoid relying too heavily on one exact orientation or crop.

Validation and test images use the same resize/crop steps but no random augmentation.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 12. Create a simple dataset class

The regular training data comes from folders, but our validation and test data are now represented by explicit file lists.

This small dataset class lets us use those lists directly and keeps the label mapping the same everywhere.

In [ ]:
class PlantDataset(Dataset):
    def __init__(self, items, class_to_idx, transform=None):
        self.items = items
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, class_name = self.items[index]

        with Image.open(path) as img:
            image = img.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = self.class_to_idx[class_name]
        return image, label


class_to_idx = {name: i for i, name in enumerate(class_names)}

train_dataset = PlantDataset(train_items, class_to_idx, train_transform)
validation_dataset = PlantDataset(validation_items, class_to_idx, eval_transform)
test_dataset = PlantDataset(test_items, class_to_idx, eval_transform)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", len(class_names))

## 13. Create the data loaders

The data loaders take care of batching the images and feeding them to the GPU.

The training loader shuffles the training examples. Validation and test loaders do not shuffle because their order is not important for evaluation and keeping them fixed makes debugging easier.

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

## 14. Look at one training batch

Before the long training runs, I want one quick check that the data loader is returning tensors with the expected shape.

The model will receive batches shaped like:

`batch size × 3 × 224 × 224`

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First five labels:", labels[:5].tolist())

## 15. Function for calculating evaluation metrics

For a multi-class problem with 38 classes, accuracy alone does not tell the whole story.

We will record accuracy, macro precision, macro recall and macro F1. Macro averaging gives every class equal weight when calculating the average.

In [ ]:
def calculate_metrics(true_labels, predicted_labels):
    return {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "precision": precision_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),
        "recall": recall_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),
        "f1": f1_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),
    }

## 16. Train and evaluate one model

This is the main training loop.

The model learns from the training set. After every epoch, we run it on the validation set without updating the weights.

The best checkpoint is chosen using **validation macro F1**. That checkpoint will later be tested only once on the untouched test subset.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()

    running_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=device.type == "cuda",
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.detach().cpu().numpy())
        predicted_labels.extend(predictions.detach().cpu().numpy())

    metrics = calculate_metrics(true_labels, predicted_labels)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics


@torch.no_grad()
def evaluate_model(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    true_labels = []
    predicted_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.cpu().numpy())
        predicted_labels.extend(predictions.cpu().numpy())

    metrics = calculate_metrics(true_labels, predicted_labels)
    metrics["loss"] = running_loss / len(loader.dataset)

    return metrics, true_labels, predicted_labels


def fit_model(model, model_name):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=1,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
    )

    best_f1 = -1.0
    best_epoch = 0
    patience_counter = 0

    history = []

    checkpoint_path = ARTIFACT_DIR / f"{model_name}_best.pth"

    for epoch in range(1, EPOCHS + 1):
        start = time.time()

        train_metrics = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
        )

        val_metrics, _, _ = evaluate_model(
            model,
            validation_loader,
            criterion,
        )

        scheduler.step(val_metrics["loss"])

        row = {
            "epoch": epoch,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_precision": train_metrics["precision"],
            "train_recall": train_metrics["recall"],
            "train_f1": train_metrics["f1"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
        }

        history.append(row)

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train acc {train_metrics['accuracy']:.4f} | "
            f"val acc {val_metrics['accuracy']:.4f} | "
            f"val F1 {val_metrics['f1']:.4f} | "
            f"time {time.time() - start:.1f}s"
        )

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "class_names": class_names,
                    "model_name": model_name,
                    "image_size": IMAGE_SIZE,
                    "mean": IMAGENET_MEAN,
                    "std": IMAGENET_STD,
                    "best_val_f1": best_f1,
                    "best_epoch": best_epoch,
                },
                checkpoint_path,
            )
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

    history = pd.DataFrame(history)

    history.to_csv(
        ARTIFACT_DIR / f"{model_name}_history.csv",
        index=False,
    )

    print(
        f"Best validation F1: {best_f1:.4f} "
        f"at epoch {best_epoch}"
    )

    return history, checkpoint_path

## 17. Build the ResNet-50 model

ResNet-50 is a pretrained convolutional neural network. We keep its learned ImageNet feature extractor and replace the final classification layer so that it predicts our 38 plant classes.

The pretrained weights give the model a useful visual starting point instead of learning everything from zero.

In [ ]:
def build_resnet50():
    model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(class_names))
    return model


resnet50 = build_resnet50()

print("ResNet-50 ready.")
print("Output classes:", len(class_names))

## 18. Train ResNet-50

This is the first full training run.

The model is trained only on `train/`. The validation set is used to monitor progress and select the best checkpoint.

In [ ]:
resnet_history, resnet_checkpoint = fit_model(
    resnet50,
    "resnet50",
)

## 19. Plot the ResNet-50 learning curves

These plots help us see whether the model is learning normally and whether the training and validation curves begin to separate.

A widening gap is something we will pay attention to because it can indicate overfitting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(resnet_history["epoch"], resnet_history["train_accuracy"], label="Train accuracy")
ax.plot(resnet_history["epoch"], resnet_history["val_accuracy"], label="Validation accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("ResNet-50 Accuracy")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(resnet_history["epoch"], resnet_history["train_loss"], label="Train loss")
ax.plot(resnet_history["epoch"], resnet_history["val_loss"], label="Validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("ResNet-50 Loss")
ax.legend()
plt.show()

## 20. Build and train EfficientNet-B0

EfficientNet-B0 gives us a second pretrained architecture with a different design from ResNet-50.

We use the same dataset, preprocessing and evaluation procedure so that the comparison is fair.

In [ ]:
def build_efficientnet_b0():
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        len(class_names),
    )
    return model


efficientnet_b0 = build_efficientnet_b0()

print("EfficientNet-B0 ready.")
print("Output classes:", len(class_names))

## 21. Train EfficientNet-B0

The second model now goes through the same training process.

We will compare the two models using the same validation metrics instead of choosing a model from the final test set.

In [ ]:
efficientnet_history, efficientnet_checkpoint = fit_model(
    efficientnet_b0,
    "efficientnet_b0",
)

## 22. Plot the EfficientNet-B0 learning curves

These plots are saved in the notebook as part of the project record and can also be used later in the report.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(efficientnet_history["epoch"], efficientnet_history["train_accuracy"], label="Train accuracy")
ax.plot(efficientnet_history["epoch"], efficientnet_history["val_accuracy"], label="Validation accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("EfficientNet-B0 Accuracy")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(efficientnet_history["epoch"], efficientnet_history["train_loss"], label="Train loss")
ax.plot(efficientnet_history["epoch"], efficientnet_history["val_loss"], label="Validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("EfficientNet-B0 Loss")
ax.legend()
plt.show()

## 23. Compare the two models on validation data

This is where we make the model-selection decision.

The test set has not been used here. We compare the two best checkpoints on the validation set and record the main metrics side by side.

In [ ]:
def load_checkpoint(path, model):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return checkpoint


resnet50 = build_resnet50()
resnet_checkpoint_data = load_checkpoint(resnet_checkpoint, resnet50)

efficientnet_b0 = build_efficientnet_b0()
efficientnet_checkpoint_data = load_checkpoint(
    efficientnet_checkpoint,
    efficientnet_b0,
)

criterion = nn.CrossEntropyLoss()

resnet_val_metrics, _, _ = evaluate_model(
    resnet50,
    validation_loader,
    criterion,
)

efficientnet_val_metrics, _, _ = evaluate_model(
    efficientnet_b0,
    validation_loader,
    criterion,
)

validation_comparison = pd.DataFrame([
    {
        "model": "ResNet-50",
        "accuracy": resnet_val_metrics["accuracy"],
        "precision": resnet_val_metrics["precision"],
        "recall": resnet_val_metrics["recall"],
        "f1": resnet_val_metrics["f1"],
    },
    {
        "model": "EfficientNet-B0",
        "accuracy": efficientnet_val_metrics["accuracy"],
        "precision": efficientnet_val_metrics["precision"],
        "recall": efficientnet_val_metrics["recall"],
        "f1": efficientnet_val_metrics["f1"],
    },
])

display(validation_comparison)

validation_comparison.to_csv(
    ARTIFACT_DIR / "validation_model_comparison.csv",
    index=False,
)

best_model_name = validation_comparison.sort_values(
    "f1",
    ascending=False,
).iloc[0]["model"]

print("Model selected from validation macro F1:", best_model_name)

## 24. Evaluate the selected model on the untouched test subset

Now the development decisions are finished.

The selected model is evaluated on the test images that were kept aside from the start. We do not change the model based on this result.

These are the numbers we can use as the final hold-out evaluation for the notebook.

In [ ]:
if best_model_name == "ResNet-50":
    final_model = resnet50
    final_checkpoint = resnet_checkpoint
else:
    final_model = efficientnet_b0
    final_checkpoint = efficientnet_checkpoint

final_checkpoint_data = torch.load(
    final_checkpoint,
    map_location=device,
)

final_model.load_state_dict(final_checkpoint_data["model_state_dict"])
final_model.to(device)
final_model.eval()

test_metrics, test_true, test_pred = evaluate_model(
    final_model,
    test_loader,
    criterion,
)

print("FINAL TEST RESULTS")
print("=" * 40)
print(f"Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall   : {test_metrics['recall']:.4f}")
print(f"Macro F1 : {test_metrics['f1']:.4f}")

pd.DataFrame([{
    "model": best_model_name,
    **test_metrics,
}]).to_csv(
    ARTIFACT_DIR / "final_test_results.csv",
    index=False,
)

## 25. Detailed classification report

The overall metrics are useful, but a disease classifier can still behave differently across individual classes.

This report gives precision, recall and F1 for every class so we can see where the model performs strongly and where it has difficulty.

In [ ]:
report = classification_report(
    test_true,
    test_pred,
    target_names=class_names,
    digits=4,
    zero_division=0,
)

print(report)

with open(ARTIFACT_DIR / "classification_report.txt", "w", encoding="utf-8") as file:
    file.write(report)

## 26. Confusion matrix

The confusion matrix shows which classes are being confused with each other.

This is particularly useful for diseases that have visually similar symptoms.

In [ ]:
cm = confusion_matrix(test_true, test_pred)

fig, ax = plt.subplots(figsize=(18, 18))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        name.replace("___", "")
        for name in class_names
    ],
)

display.plot(
    ax=ax,
    xticks_rotation=90,
    colorbar=False,
)

ax.set_title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()

fig.savefig(
    ARTIFACT_DIR / "final_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight",
)

## 27. Grad-CAM helper

Grad-CAM gives us a visual explanation for a prediction.

The basic idea is simple: we take the feature maps from a late convolutional layer, look at how strongly the chosen class depends on those feature maps, and use that information to create a heatmap.

The heatmap is an explanation of the model's attention for that prediction. It is not a ground-truth disease segmentation mask.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = target_layer.register_forward_hook(
            self.save_activations
        )
        self.backward_handle = target_layer.register_full_backward_hook(
            self.save_gradients
        )

    def save_activations(self, module, inputs, output):
        self.activations = output.detach()

    def save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(self, image_tensor, class_index=None):
        self.model.eval()

        self.model.zero_grad(set_to_none=True)

        output = self.model(image_tensor)

        if class_index is None:
            class_index = int(output.argmax(dim=1).item())

        score = output[0, class_index]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)

        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().numpy()

        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        probability = torch.softmax(output, dim=1)[0, class_index].item()

        return cam, class_index, probability

## 28. Connect Grad-CAM to the selected model

ResNet-50 and EfficientNet-B0 have different internal layouts, so the late convolutional layer is selected differently for each architecture.

We will use the last suitable convolutional feature block before the final classifier.

In [ ]:
if best_model_name == "ResNet-50":
    gradcam_layer = final_model.layer4[-1]
else:
    gradcam_layer = final_model.features[-1]

gradcam = GradCAM(final_model, gradcam_layer)

print("Grad-CAM target layer:", gradcam_layer)

## 29. Run Grad-CAM on one reserved demo image

This cell uses one of the 76 images that was held aside specifically for the live demonstration.

The image is not used for training, validation or testing. We are only running inference on it here.

In [ ]:
demo_row = demo_manifest.iloc[0]

demo_path = Path(demo_row["demo_path"])
demo_class = demo_row["class_name"]

with Image.open(demo_path) as img:
    original_image = img.convert("RGB")

input_tensor = eval_transform(original_image).unsqueeze(0).to(device)

cam, predicted_index, confidence = gradcam.generate(input_tensor)

predicted_class = class_names[predicted_index]

print("Actual reserved class :", demo_class)
print("Model prediction      :", predicted_class)
print(f"Confidence            : {confidence:.4f}")

## 30. Show the original image and Grad-CAM overlay

The left image is the original leaf. The right image adds the Grad-CAM heatmap.

The highlighted regions show where the selected prediction drew the strongest contribution from the model's final convolutional features.

In [ ]:
from PIL import Image, ImageFilter

# Resize the Grad-CAM map smoothly to the image size
cam_image = Image.fromarray(
    np.uint8(cam * 255)
).resize(
    (IMAGE_SIZE, IMAGE_SIZE),
    Image.Resampling.BICUBIC,
).filter(
    ImageFilter.GaussianBlur(radius=2)
)

cam_smooth = np.asarray(cam_image, dtype=np.float32) / 255.0

original_array = np.array(
    original_image.resize((IMAGE_SIZE, IMAGE_SIZE))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(original_array)
axes[0].set_title(
    f"Original\n{demo_class.replace('___', ' - ')}"
)
axes[0].axis("off")

axes[1].imshow(original_array)
axes[1].imshow(
    cam_smooth,
    cmap="jet",
    alpha=0.45,
    interpolation="bilinear",
)
axes[1].set_title(
    f"Grad-CAM\n"
    f"{predicted_class.replace('___', ' - ')} "
    f"({confidence:.2%})"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import shutil

working_dir = Path("/kaggle/working")
package_dir = working_dir / "project_outputs"

# Remove an older package if it exists
if package_dir.exists():
    shutil.rmtree(package_dir)

package_dir.mkdir()

# Copy the important project outputs
shutil.copytree(
    working_dir / "artifacts",
    package_dir / "artifacts",
)

shutil.copytree(
    working_dir / "demo_images",
    package_dir / "demo_images",
)

# Create one ZIP containing everything
project_zip = shutil.make_archive(
    str(working_dir / "project_outputs"),
    "zip",
    root_dir=working_dir,
    base_dir="project_outputs",
)

print("Project package created:")
print(project_zip)
print(
    f"Size: "
    f"{Path(project_zip).stat().st_size / (1024 * 1024):.2f} MB"
)

## 31. Save the final model package for later use

The website will need more than the model weights. It also needs the class names and preprocessing information.

This cell saves those details in one small JSON file alongside the final checkpoint.

In [ ]:
final_model_path = ARTIFACT_DIR / "plant_disease_model.pth"

shutil.copy2(
    final_checkpoint,
    final_model_path,
)

model_config = {
    "model_name": best_model_name,
    "num_classes": len(class_names),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
}

with open(
    ARTIFACT_DIR / "model_config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(model_config, file, indent=2)

print("Final model:", final_model_path)
print("Model config:", ARTIFACT_DIR / "model_config.json")

## 32. Final project artifacts

The files in `/kaggle/working/artifacts/` are the useful outputs from this notebook.

They include the model checkpoint, training histories, split manifests, comparison results, classification report and confusion matrix.

The `demo_images.zip` file is kept separately because those 76 images are specifically for the live website demonstration.

In [ ]:
print("Artifacts created:")
for file in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", file.name)

print("\nDemo files:")
print(" -", DEMO_DIR)
print(" -", demo_zip)

## 33. Notes for the report

The final report should use the actual numbers produced by this notebook rather than assumed numbers.

Important points to document:

- The dataset contains 38 classes.
- The provided training and validation counts were audited before training.
- Two validation images per class were reserved as a separate live-demo set.
- The remaining validation images were divided into validation and final hold-out test subsets.
- ResNet-50 and EfficientNet-B0 were trained using pretrained ImageNet weights.
- Model selection was based on validation performance.
- The final selected model was evaluated on an untouched test subset.
- Grad-CAM was used for visual explainability.
- Grad-CAM is an explanation method and should not be described as a ground-truth segmentation mask.